# Preference data and reward models

LLM Training Lab, module 17. Regime PREFERENCE: SmolLM2-360M plus a scalar head, every weight trained with the Bradley-Terry loss on UltraFeedback preference pairs.

Run the cells in order on one T4 (Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2). Steps 1 to 4 also run on a CPU. Code cell numbers match the lab page.

In [ ]:
import os, sys, platform, shutil, subprocess
from importlib.metadata import version, PackageNotFoundError
os.environ["CUDA_VISIBLE_DEVICES"] = "0"            # one GPU, set before torch loads
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
problems = []
if sys.version_info < (3, 11):
    problems.append("Python 3.11 or newer is needed (3.12 recommended). Start a current Colab or Kaggle session.")
try:
    smi = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:                                   # a CPU runtime has no nvidia-smi
    smi = []
gpu = smi[0] if smi else ""
if not gpu:
    problems.append("No GPU: steps 1 to 4 still run. For step 5 on, Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2.")
elif "P100" in gpu:
    problems.append("A P100 has no kernels in current PyTorch. Pick a T4.")
cache = os.path.expanduser("~/.cache")
os.makedirs(cache, exist_ok=True)
free_disk = shutil.disk_usage(cache).free / 1e9
if free_disk < 6:
    problems.append("Free at least 6 GB of disk for the model and data downloads (" + str(round(free_disk)) + " GB free).")
try:
    torch_version = version("torch")
except PackageNotFoundError:
    torch_version = "not installed"
print("Python", platform.python_version(), "| GPU", gpu or "none", "| torch", torch_version, "| disk free", round(free_disk), "GB")
for p in problems:
    print("PROBLEM", p)
print("Runtime OK" if not problems else "Read the PROBLEM lines first.")

In [ ]:
# Pinned Hugging Face stack of our recorded run. torch and numpy stay as the runtime ships them.
%pip uninstall -y -q torchao
%pip install -q transformers==5.22.1 trl==1.17.1 accelerate==1.15.0 datasets==5.1.0 tokenizers==0.23.3 huggingface_hub==2.4.0

## Step 1. Turn a preference into a probability (NumPy)

Code cell 1 on the lab page.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def bt_loss(r_chosen, r_rejected):
    # -log sigmoid(r_chosen - r_rejected), averaged over pairs. logaddexp(0, -d) never overflows.
    d = np.asarray(r_chosen, dtype=float) - np.asarray(r_rejected, dtype=float)
    return float(np.mean(np.logaddexp(0, -d)))

def bt_grad(r_chosen, r_rejected):
    # gradient of the mean loss with respect to each chosen and each rejected reward
    d = np.asarray(r_chosen, dtype=float) - np.asarray(r_rejected, dtype=float)
    g = -(1 - sigmoid(d)) / len(d)
    return g, -g

print("P(chosen preferred) at margin 0.5:", round(float(sigmoid(0.5)), 4), "| loss", round(bt_loss([0.5], [0.0]), 4))
print("loss at margin 0:", round(bt_loss([0, 0], [0, 0]), 4), "| ln 2 =", round(float(np.log(2)), 4))
print("add 7 to both rewards:", round(bt_loss([7.5], [7.0]), 4), "| scale the margin by 4:", round(bt_loss([2.0], [0.0]), 4))

rc, rr = np.array([1.2, -0.3, 0.4]), np.array([0.2, 0.5, 0.4])
gc, gr = bt_grad(rc, rr)
eps = 1e-6
num = [(bt_loss(rc + eps * np.eye(3)[i], rr) - bt_loss(rc - eps * np.eye(3)[i], rr)) / (2 * eps) for i in range(3)]
print("analytic dL/dr_chosen", np.round(gc, 5), "| finite difference", np.round(num, 5))
print("gradient sums to zero per pair:", bool(np.allclose(gc + gr, 0)))

## Step 2. Put a scalar head on SmolLM2-360M

Code cell 2 on the lab page.

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # Kaggle has 2 GPUs; this lab uses one
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

torch.set_num_threads(4)
BASE = "HuggingFaceTB/SmolLM2-360M"
tok = AutoTokenizer.from_pretrained(BASE)
tok.chat_template = AutoTokenizer.from_pretrained(BASE + "-Instruct").chat_template   # the base ships none
tok.pad_token = "<|endoftext|>"                    # padding stays apart from the <|im_end|> that closes a turn

torch.manual_seed(0)
rm = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=1, dtype=torch.float32).eval()
print(rm.score, "| parameters", sum(p.numel() for p in rm.parameters()))

def chat_ids(prompt, response):
    return list(tok.apply_chat_template([{"role": "user", "content": prompt},
                                         {"role": "assistant", "content": response}])["input_ids"])

a = chat_ids("Name a fruit.", "A mango.")
b = chat_ids("Name a fruit.", "A mango is a stone fruit grown in South Asia.")
ids = torch.full((2, len(b)), tok.pad_token_id)
ids[0, :len(a)], ids[1, :len(b)] = torch.tensor(a), torch.tensor(b)
mask = (ids != tok.pad_token_id).long()
print("rows of", len(a), "and", len(b), "tokens, last token of row 0:", repr(tok.decode(a[-1:])))

try:
    rm(input_ids=ids, attention_mask=mask)
except ValueError as e:
    print("ValueError:", e)
rm.config.pad_token_id = tok.pad_token_id
with torch.no_grad():
    rewards = rm(input_ids=ids, attention_mask=mask).logits.squeeze(-1)
    h = rm.model(input_ids=ids, attention_mask=mask).last_hidden_state
    last = mask.sum(-1) - 1                         # the last real token of each right-padded row
    by_hand = rm.score(h[torch.arange(2), last]).squeeze(-1)
print("rewards", rewards.tolist(), "| by hand", by_hand.tolist(), "| equal", torch.allclose(rewards, by_hand))

## Step 3. Load the preference pairs and look at what they reward

Code cell 3 on the lab page.

In [ ]:
from datasets import load_dataset
import numpy as np

MAXLEN = 1024

def select_pairs(split, n, shuffle=True):
    # Shuffle with seed 0 (training) or keep file order (held out); keep pairs whose two sequences fit MAXLEN tokens
    ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split=split)
    if shuffle:
        ds = ds.shuffle(seed=0)
    out, read = [], 0
    for ex in ds:
        read += 1
        c, r = ex["chosen"][-1]["content"], ex["rejected"][-1]["content"]
        if c == r:
            continue
        ci, ri = chat_ids(ex["prompt"], c), chat_ids(ex["prompt"], r)
        if len(ci) <= MAXLEN and len(ri) <= MAXLEN:
            out.append({"prompt": ex["prompt"], "chosen": c, "rejected": r, "c": ci, "r": ri,
                        "sc": ex["score_chosen"], "sr": ex["score_rejected"]})
        if len(out) == n:
            return out, read

train, read_train = select_pairs("train_prefs", 3000)
held, read_held = select_pairs("test_prefs", 1000, shuffle=False)
for name, pairs, read in [("train", train, read_train), ("held-out", held, read_held)]:
    longer = np.mean([len(p["c"]) > len(p["r"]) for p in pairs])
    ties = np.mean([p["sc"] == p["sr"] for p in pairs])
    mean_len = np.mean([len(p["c"]) + len(p["r"]) for p in pairs]) / 2
    print(name, "| kept", len(pairs), "of", read, "read | chosen longer", round(float(longer), 3),
          "| equal GPT-4 scores", round(float(ties), 3), "| mean tokens", round(float(mean_len)))
print("length-only rule on held-out, pick the longer reply:", round(float(np.mean([len(p["c"]) > len(p["r"]) for p in held])), 3))

## Step 4. Match RewardTrainer's loss on the same batch

Code cell 4 on the lab page.

In [ ]:
from datasets import Dataset
from trl import RewardConfig, RewardTrainer

def as_chat(prompt, reply):
    return [{"role": "user", "content": prompt}, {"role": "assistant", "content": reply}]

rows = [{"chosen": as_chat(p["prompt"], p["chosen"]), "rejected": as_chat(p["prompt"], p["rejected"])} for p in train[:8]]
args = RewardConfig(output_dir="rm-check", report_to="none", use_cpu=True, bf16=False, max_length=MAXLEN,
                    per_device_train_batch_size=4)
trainer = RewardTrainer(model=rm, args=args, processing_class=tok, train_dataset=Dataset.from_list(rows))
print("same token ids as yours:", all(list(trainer.train_dataset[i]["chosen_ids"]) == train[i]["c"] and
                                      list(trainer.train_dataset[i]["rejected_ids"]) == train[i]["r"] for i in range(8)))
batch = next(iter(trainer.get_train_dataloader()))
n = batch["input_ids"].shape[0] // 2                   # TRL stacks the chosen rows first, then the rejected rows
with torch.no_grad():
    r = rm(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits.squeeze(-1)
    own = -torch.nn.functional.logsigmoid(r[:n] - r[n:]).mean().item()
    lib = trainer.compute_loss(rm, dict(batch)).item()
print("batch", list(batch["input_ids"].shape), "| own", own, "| TRL", lib, "| difference", abs(own - lib))
print("rewards chosen", [round(x, 3) for x in r[:n].tolist()], "| rejected", [round(x, 3) for x in r[n:].tolist()])

## Step 5. Train the reward model on a T4 (15 to 17 min of training; RUN_TRAINING = False loads ours)

Code cell 5 on the lab page.

In [ ]:
import math, random, time
GPU = torch.device("cuda")
RUN_TRAINING = True       # False loads our seed-0 reward model from the Hub instead (about 1 min)

def pad(seqs):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), tok.pad_token_id)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        mask[i, :len(s)] = 1
    return ids.to(GPU), mask.to(GPU)

def batch_plan(pairs, size, seed):
    # shuffle, sort each chunk of 64 pairs by length, cut into batches, shuffle the batches: less padding, same data
    rng = random.Random(seed)
    order = list(range(len(pairs)))
    rng.shuffle(order)
    plan = []
    for k in range(0, len(order), 64):
        chunk = sorted(order[k:k + 64], key=lambda i: -max(len(pairs[i]["c"]), len(pairs[i]["r"])))
        plan += [chunk[j:j + size] for j in range(0, len(chunk), size)]
    rng.shuffle(plan)
    return plan

def train_rm(pairs, seed=0, lr=1e-5, size=8):
    torch.manual_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(BASE, num_labels=1, dtype=torch.float32).to(GPU)
    model.config.pad_token_id = tok.pad_token_id
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.config.use_cache = False
    plan = batch_plan(pairs, size, seed)
    warm = max(1, int(0.03 * len(plan)))
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else
                                              0.5 * (1 + math.cos(math.pi * min(1.0, (s - warm) / (len(plan) - warm)))))
    scaler = torch.amp.GradScaler("cuda", init_scale=2048)    # fp16 gradients need loss scaling on a T4
    model.train()
    torch.cuda.reset_peak_memory_stats()
    t0, hist = time.time(), []
    for step, idx in enumerate(plan):
        chunk = [pairs[i] for i in idx]
        ids, mask = pad([p["c"] for p in chunk] + [p["r"] for p in chunk])
        with torch.autocast("cuda", dtype=torch.float16):
            r = model(input_ids=ids, attention_mask=mask).logits.squeeze(-1).float()
        margin = r[:len(chunk)] - r[len(chunk):]
        loss = -torch.nn.functional.logsigmoid(margin).mean()
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        sched.step()
        hist.append((loss.item(), (margin > 0).float().mean().item()))
        if (step + 1) % 75 == 0 or step == 0:
            recent = hist[-75:]
            print("step", step + 1, "of", len(plan), "| loss", round(sum(h[0] for h in recent) / len(recent), 4),
                  "| batch accuracy", round(sum(h[1] for h in recent) / len(recent), 3), "| seconds", round(time.time() - t0))
    print("peak GB", round(torch.cuda.max_memory_allocated() / 1e9, 2))
    return model.eval()

if RUN_TRAINING:
    rm = train_rm(train, seed=0)
else:
    rm = AutoModelForSequenceClassification.from_pretrained("aimlcompanion/llm-training-lab", subfolder="lltRewardModels/rm-s0",
                                                            dtype=torch.float32).to(GPU).eval()

## Step 6. Read held-out accuracy honestly

Code cell 6 on the lab page.

In [ ]:
def score(model, seqs, budget=16384):
    # one reward per token-id list; batches of similar length, at most 32 rows or 16,384 tokens each
    out, order, k = [None] * len(seqs), sorted(range(len(seqs)), key=lambda i: -len(seqs[i])), 0
    while k < len(order):
        idx = order[k:k + max(1, min(32, budget // len(seqs[order[k]])))]
        ids, mask = pad([seqs[i] for i in idx])
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            r = model(input_ids=ids, attention_mask=mask).logits.squeeze(-1).float().tolist()
        for i, v in zip(idx, r):
            out[i] = v
        k += len(idx)
    return out

def held_out_report(model, pairs):
    s = score(model, [p["c"] for p in pairs] + [p["r"] for p in pairs])
    rc, rr = np.array(s[:len(pairs)]), np.array(s[len(pairs):])
    win = rc > rr
    longer = np.array([len(p["c"]) > len(p["r"]) for p in pairs])
    shorter = np.array([len(p["c"]) < len(p["r"]) for p in pairs])
    tie = np.array([p["sc"] == p["sr"] for p in pairs])
    boot = np.random.default_rng(0).integers(0, len(win), (10000, len(win)))
    lo, hi = np.percentile(win[boot].mean(1), [2.5, 97.5])
    lengths = np.array([len(p["c"]) for p in pairs] + [len(p["r"]) for p in pairs])
    print("accuracy", round(win.mean(), 3), "95% CI", round(lo, 3), "to", round(hi, 3),
          "| without equal-score pairs", round(win[~tie].mean(), 3))
    print("chosen longer", round(win[longer].mean(), 3), "of", longer.sum(), "| chosen shorter", round(win[shorter].mean(), 3),
          "of", shorter.sum(), "| length-only rule", round(longer.mean(), 3))
    print("correlation of reward with length", round(np.corrcoef(np.concatenate([rc, rr]), lengths)[0, 1], 3))
    return win

win_s0 = held_out_report(rm, held)

## Step 7. Test the length bias (RUN_EXPERIMENT = True trains two more models, about 35 min)

Code cell 7 on the lab page.

In [ ]:
import gc
RUN_EXPERIMENT = False    # True trains two more reward models on your T4; our runs are in the table of this step

def select_balanced(n):
    # same shuffled stream as step 3, but half the pairs have the longer reply chosen and half the shorter one
    ds = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs").shuffle(seed=0)
    out, count = [], {True: 0, False: 0}
    for ex in ds:
        c, r = ex["chosen"][-1]["content"], ex["rejected"][-1]["content"]
        if c == r:
            continue
        ci, ri = chat_ids(ex["prompt"], c), chat_ids(ex["prompt"], r)
        if len(ci) > MAXLEN or len(ri) > MAXLEN or len(ci) == len(ri) or count[len(ci) > len(ri)] == n // 2:
            continue
        count[len(ci) > len(ri)] += 1
        out.append({"prompt": ex["prompt"], "chosen": c, "rejected": r, "c": ci, "r": ri,
                    "sc": ex["score_chosen"], "sr": ex["score_rejected"]})
        if len(out) == n:
            return out

if RUN_EXPERIMENT:
    rm_cpu = rm.cpu()                         # keep step 5's model, but off the GPU
    gc.collect()
    torch.cuda.empty_cache()
    for name, pairs, seed in [("seed 1", train, 1), ("length-balanced, seed 0", select_balanced(3000), 0)]:
        m = train_rm(pairs, seed=seed)
        print(name)
        held_out_report(m, held)
        del m
        gc.collect()
        torch.cuda.empty_cache()
    rm = rm_cpu.to(GPU)

## Step 8. Best-of-n against a gold reward model

Code cell 8 on the lab page.

In [ ]:
import json
from huggingface_hub import hf_hub_download

# our pool: 64 held-out prompts x 64 samples from SmolLM2-360M-Instruct, with the gold reward of each sample
pool = json.load(open(hf_hub_download("aimlcompanion/llm-training-lab", "lltRewardModels/bon_pool.json")))
P, N = len(pool["prompts"]), pool["n"]
proxy = np.array(score(rm, [chat_ids(pool["prompts"][s["p"]]["prompt"], s["text"]) for s in pool["samples"]])).reshape(P, N)
gold = np.array(pool["gold"]).reshape(P, N)
tokens = np.array([s["newTokens"] for s in pool["samples"]]).reshape(P, N)

def best_of_n(select, value, n):
    # expected value of the sample that select ranks first among n drawn without replacement from the N recorded ones
    total = 0.0
    for p in range(len(select)):
        order = np.argsort(select[p])                       # ascending: rank i has i samples below it
        weight = np.array([math.comb(i, n - 1) for i in range(N)], dtype=float) / math.comb(N, n)
        total += float((weight * value[p][order]).sum())
    return total / len(select)

print("n    KL nats   proxy    gold     new tokens")
for n in [1, 2, 4, 8, 16, 32, 64]:
    kl = math.log(n) - (n - 1) / n
    print(n, round(kl, 3), round(best_of_n(proxy, proxy, n), 3), round(best_of_n(proxy, gold, n), 3),
          round(best_of_n(proxy, tokens, n), 1), sep="    ")
print("gold picking by gold, n = 64:", round(best_of_n(gold, gold, 64), 3))

## Step 9. Score a RewardBench 2 subset

Code cell 9 on the lab page.

In [ ]:
import random

def rb2_subset(per_subset=60, seed=0):
    # 60 items from each of the five subsets that score one chosen reply against three rejected ones
    ds = load_dataset("allenai/reward-bench-2", split="test")
    rng, items = random.Random(seed), []
    for sub in ["Factuality", "Precise IF", "Math", "Safety", "Focus"]:
        idx = [i for i, s in enumerate(ds["subset"]) if s == sub]
        items += [dict(ds[i]) for i in sorted(rng.sample(idx, per_subset))]
    return items

items = rb2_subset()
seqs = [chat_ids(it["prompt"], reply)[:2048] for it in items for reply in it["chosen"][:1] + it["rejected"][:3]]
s = np.array(score(rm, seqs)).reshape(len(items), 4)
correct = s[:, 0] > s[:, 1:].max(1)
for sub in ["Factuality", "Precise IF", "Math", "Safety", "Focus"]:
    print(sub, round(float(np.mean([c for c, it in zip(correct, items) if it["subset"] == sub])), 3))
print("mean of the five", round(float(correct.mean()), 3), "| random guessing 0.25")